# 02 - Baseline Model

**Purpose:** Calculate baseline operating costs without battery storage for comparison.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config.settings import load_config
from src.data.loaders import generate_synthetic_data
from src.evaluation.baseline import compute_baseline_cost

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Load Data

In [ ]:
settings = load_config('../config/config.yaml')

df = generate_synthetic_data(
    start_date=settings.start_date,
    end_date=settings.end_date,
    resolution_minutes=settings.resolution_minutes,
    solar_capacity_kw=settings.solar_capacity_kw,
    timezone=settings.timezone,
)
print(f"Loaded {len(df)} time steps")

## Baseline Scenario: No Battery

In the baseline scenario:
- Excess solar is exported at feed-in tariff
- Deficit is imported at spot price
- No load shifting or price arbitrage

In [ ]:
dt_hours = settings.resolution_minutes / 60.0

baseline = compute_baseline_cost(
    prices=df['price'].values,
    p_solar=df['solar'].values,
    p_load=df['load'].values,
    feed_in_tariff=settings.grid.feed_in_tariff,
    dt_hours=dt_hours,
)

print("=" * 50)
print("BASELINE RESULTS (No Battery)")
print("=" * 50)
print(f"Total Cost:           ${baseline.total_cost:.2f}")
print(f"Grid Import:          {baseline.total_import_kwh:.1f} kWh")
print(f"Grid Export:          {baseline.total_export_kwh:.1f} kWh")
print(f"Peak Import:          {baseline.peak_import_kw:.2f} kW")
print(f"Self-Consumption:     {baseline.self_consumption_ratio*100:.1f}%")
print("=" * 50)

## Visualize Baseline Power Flows

In [ ]:
# Calculate baseline power flows
net_load = df['load'].values - df['solar'].values
p_import = np.maximum(net_load, 0)
p_export = np.maximum(-net_load, 0)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# First day only
day_idx = slice(0, 24)
t = df.index[day_idx]

# Generation and load
axes[0].fill_between(t, df['solar'].iloc[day_idx], alpha=0.7, label='Solar', color='gold')
axes[0].plot(t, df['load'].iloc[day_idx], 'k-', linewidth=2, label='Load')
axes[0].set_ylabel('Power (kW)')
axes[0].set_title('Baseline: Generation vs Load')
axes[0].legend()

# Grid flows
axes[1].fill_between(t, p_import[day_idx], alpha=0.7, label='Grid Import', color='red')
axes[1].fill_between(t, -p_export[day_idx], alpha=0.7, label='Grid Export', color='green')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Power (kW)')
axes[1].set_xlabel('Time')
axes[1].set_title('Baseline: Grid Power Flows')
axes[1].legend()

plt.tight_layout()
plt.show()

## Cost Breakdown by Time

In [ ]:
# Hourly cost calculation
hourly_import_cost = df['price'].values * p_import * dt_hours
hourly_export_revenue = settings.grid.feed_in_tariff * p_export * dt_hours
hourly_net_cost = hourly_import_cost - hourly_export_revenue

fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(df.index[:24], hourly_net_cost[:24], width=0.03, color='steelblue', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Cost ($)')
ax.set_xlabel('Time')
ax.set_title('Hourly Net Cost (Import Cost - Export Revenue)')

plt.tight_layout()
plt.show()